In [ ]:
import mesa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from mesa.space import SingleGrid
from multiprocessing.pool import ThreadPool
from pathlib import Path


In [ ]:
GRID_SIZE   = 100
NUM_GROUPS  = 4
EMPTY_CELLS  = 0.10      

SM_THRESHOLD   = 0.5         

BETA_SAME   = 0.07           
BETA_DIFF   = 0.04   

INFECT_DAYS = 7  

INIT_INF    = 0.015         

A_COEF      = 1.0          
DEFAULT_B_COEF = 1.5          
SMI_THRESHOLD = 0.5       

MAX_SCHELLING = 300
MAX_EPIDEMIC  = 300


SAVE_PART_ONE = [0, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300]
SAVE_PART_TWO  = [0, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300]

AGENT_STATES = ["healthy", "infected", "resistant"]

POSITIONS_IN_GRID = [(i, j) for i in range(GRID_SIZE) for j in range(GRID_SIZE)]


In [ ]:
# HELPERS

def get_segregation_score(grid):
    score_per_agent = []

    for agent in grid.agents:
        neighbors = grid.get_neighbors(agent.pos, moore=True, include_center=False)

        occupied_neighbors = []
        for neighbor in neighbors:
            if neighbor is not None:
                occupied_neighbors.append(neighbor)
        if not occupied_neighbors:
            continue

        same_neighbors = 0
        for neighbor in occupied_neighbors:
            if neighbor.group == agent.group:
                same_neighbors += 1

        agent_satisfaction = same_neighbors / len(occupied_neighbors)

        score_per_agent.append(agent_satisfaction)

    return np.mean(score_per_agent)


In [ ]:
# agent in a cell

class person_agent(mesa.Agent):
    def __init__(self,model, group):
        super().__init__(model)
        
        self.group = group
        self.state = AGENT_STATES[0]
        self.infected_steps = 0

    def get_part_one_schelling_agent_happiness(self):
        neighbors = self.model.grid.get_neighbors(self.pos, include_center = False, moore = True)

        occupied_neighbors = []
        for neighbor in neighbors:
            if neighbor is not None:
                occupied_neighbors.append(neighbor)

        if not occupied_neighbors:
            return True

        same_group_count = 0
        for neighbor in occupied_neighbors:
            if neighbor.group == self.group:
                same_group_count = same_group_count + 1

        perc = same_group_count / len(occupied_neighbors)
        return_value = perc >= SM_THRESHOLD
        
        return return_value

    def get_part_two_schelling_agent_happiness(self):
        neighbors = self.model.grid.get_neighbors(self.pos, include_center = False, moore = True)

        occupied_neighbors = []
        for neighbor in neighbors:
            if neighbor is not None:
                occupied_neighbors.append(neighbor)

        if not occupied_neighbors:
            return True


        
        same_group_count = 0
        for neighbor in occupied_neighbors:
            if neighbor.group == self.group:
                same_group_count = same_group_count + 1

        infection_state_count = 0
        for neighbor in occupied_neighbors:
            if neighbor.state == "infected":
                infection_state_count = infection_state_count + 1
                
                
        b_coef = getattr(self.model, "b_coef", DEFAULT_B_COEF)
        M = A_COEF * (same_group_count / len(occupied_neighbors)) - b_coef * (infection_state_count / len(occupied_neighbors))
        return_value = M >= SMI_THRESHOLD
        
        return return_value


In [ ]:
# PART 1: Schelling model segregation
class schelling_model(mesa.Model):
    def __init__(self, seed=111):
        super().__init__(seed=seed)

        self.grid = SingleGrid(GRID_SIZE, GRID_SIZE, torus=False)
        self.total_steps = 0
        self.saved_grids = {}

        all_positions = list(POSITIONS_IN_GRID)
        self.rng.shuffle(all_positions)
        
        num_agents = int( (1 - EMPTY_CELLS) * (GRID_SIZE * GRID_SIZE)  )
        group_agent_count = num_agents // NUM_GROUPS

        groups = [group_agent_count, group_agent_count, group_agent_count, group_agent_count]
        
        for i in range(num_agents):
            random_number = self.rng.integers(0, 4)
            loops = 0
            while (groups[random_number] == 0):
                random_number = (random_number + 1) % NUM_GROUPS
                loops += 1
                if loops == NUM_GROUPS:
                    break

            groups[random_number] += -1
            agent = person_agent(self, random_number)
            self.grid.place_agent(agent, all_positions[i])

        
        self.final_seg_score = 0
        self.save()
            
        

    def save(self):
        if self.total_steps in SAVE_PART_ONE:
            self.saved_grids[self.total_steps] = self.get_array()
    
    def get_array(self):

        array = []
        for i in range(GRID_SIZE):
            
            row = []
            for j in range(GRID_SIZE):
                row.append(None)
            array.append(row)
    
        for agent in self.grid.agents:
            x, y = agent.pos
            array[y][x] = (agent.group, agent.state)
    
        return array

    def run(self):

        for i in range(MAX_SCHELLING):
            unhappy_agents = []
            for agent in self.grid.agents:
                if not agent.get_part_one_schelling_agent_happiness():
                    unhappy_agents.append(agent)

            if not unhappy_agents:
                print("All happy at step: ", self.total_steps + 1)
                self.saved_grids["final"] = self.get_array()
                break

            self.rng.shuffle(unhappy_agents)

            for agent in unhappy_agents:
                self.grid.move_to_empty(agent)

            
            self.total_steps += 1

            self.save()

            if i == MAX_SCHELLING - 1:
                print("max iter in schelling has been reached")
                
        self.final_seg_score = get_segregation_score(self.grid)
        

        

In [ ]:
# part 2: disease spreading
# one disease model; movement_enabled=False gives model A behaviour, True gives model B behaviour

class disease_spread_model(mesa.Model):

    def __init__(
        self,
        schelling_model_,
        seed=111,
        movement_enabled=False,
        b_coef=DEFAULT_B_COEF,
        initial_infected_positions_=None,
        initial_infection_type="random_population",
        initial_infection_group=0,
    ):
        super().__init__(seed=seed)
        self.grid = SingleGrid(GRID_SIZE, GRID_SIZE, torus=False)
        self.total_steps = 0
        self.movement_enabled = movement_enabled
        self.b_coef = b_coef
        self.initial_infection_type = initial_infection_type
        self.initial_infection_group = initial_infection_group

        for agent in schelling_model_.grid.agents:
            new_agent = person_agent(self, agent.group)
            new_agent.ever_infected = False
            self.grid.place_agent(new_agent, agent.pos)

        self.total_agents = len(list(self.grid.agents))
        self.agents_list = list(self.grid.agents)
        self.initial_infected_count = max(1, int(self.total_agents * INIT_INF))

        if initial_infected_positions_ is None:
            infected_agents = self.select_initial_infected_agents(
                initial_infection_type=initial_infection_type,
                initial_infection_group=initial_infection_group,
            )
            self.initial_infected_positions = [agent.pos for agent in infected_agents]
        else:
            self.initial_infected_positions = list(initial_infected_positions_)

        infected_position_set = set(self.initial_infected_positions)
        for agent in self.grid.agents:
            if agent.pos in infected_position_set:
                agent.state = "infected"
                agent.ever_infected = True
        self.statistics_per_step = []
        self.saved_grids = {}
        self.moved_last_step = 0

        self.update_stats()
        self.save()

    def has_different_group_neighbor(self, agent):
        for neighbor in self.grid.get_neighbors(agent.pos, moore=True, include_center=False):
            if neighbor.group != agent.group:
                return True
        return False

    def select_initial_infected_agents(self, initial_infection_type="random_population", initial_infection_group=0):
        if initial_infection_type == "random_population":
            candidates = list(self.agents_list)
        elif initial_infection_type == "single_group":
            candidates = [agent for agent in self.agents_list if agent.group == initial_infection_group]
        elif initial_infection_type == "group_boundary":
            candidates = [agent for agent in self.agents_list if self.has_different_group_neighbor(agent)]
        else:
            raise ValueError(
                "initial_infection_type must be one of: random_population, single_group, group_boundary"
            )

        if len(candidates) < self.initial_infected_count:
            candidates = list(self.agents_list)

        return list(self.rng.choice(candidates, size=self.initial_infected_count, replace=False))

    def save(self):
        if self.total_steps in SAVE_PART_TWO:
            self.saved_grids[self.total_steps] = self.get_array()

    def get_array(self):
        array = []
        for i in range(GRID_SIZE):
            row = []
            for j in range(GRID_SIZE):
                row.append(None)
            array.append(row)

        for agent in self.grid.agents:
            x, y = agent.pos
            array[y][x] = (agent.group, agent.state)

        return array

    def get_mixing_stats(self):
        mixed_neighbor_agents = 0
        cross_group_infected_neighbor_agents = 0
        for agent in list(self.grid.agents):
            has_different_group_neighbor = False
            has_different_group_infected_neighbor = False

            for neighbor in self.grid.get_neighbors(agent.pos, moore=True, include_center=False):
                if neighbor.group != agent.group:
                    has_different_group_neighbor = True
                    if neighbor.state == "infected":
                        has_different_group_infected_neighbor = True

            if has_different_group_neighbor:
                mixed_neighbor_agents += 1
            if has_different_group_infected_neighbor:
                cross_group_infected_neighbor_agents += 1

        return {
            "mixed_neighbor_agents": mixed_neighbor_agents,
            "cross_group_infected_neighbor_agents": cross_group_infected_neighbor_agents,
        }

    def update_stats(self):
        infected_agents = 0
        ever_infected_agents = 0

        for agent in list(self.grid.agents):
            if agent.state == "infected":
                infected_agents += 1
            if getattr(agent, "ever_infected", False):
                ever_infected_agents += 1

        mixing_stats = self.get_mixing_stats()
        self.statistics_per_step.append({
            "step": self.total_steps,
            "infected": infected_agents,
            "ever_infected": ever_infected_agents,
            "moved": self.moved_last_step,
            **mixing_stats,
        })

    def is_there_epidemic(self):
        for agent in self.grid.agents:
            if agent.state == "infected":
                return True
        return False

    def spread_disease_synchronously(self):
        newly_infected_agents = []

        for agent in list(self.grid.agents):
            if agent.state != "healthy":
                continue

            neighbors = self.grid.get_neighbors(agent.pos, moore=True, include_center=False)
            k_same = 0
            k_diff = 0

            for neighbor in neighbors:
                if neighbor.state == "infected":
                    if neighbor.group == agent.group:
                        k_same += 1
                    else:
                        k_diff += 1

            infection_probability = 1 - ((1 - BETA_DIFF) ** k_diff) * ((1 - BETA_SAME) ** k_same)

            if self.rng.random() < infection_probability:
                newly_infected_agents.append(agent)

        for agent in newly_infected_agents:
            agent.state = "infected"
            agent.ever_infected = True
            agent.infected_steps = 0


    def move_unhappy_agents(self):
        unhappy_agents = []
        for agent in self.grid.agents:
            if agent.state == "infected":
                continue

            if not agent.get_part_two_schelling_agent_happiness():
                unhappy_agents.append(agent)

        self.rng.shuffle(unhappy_agents)
        for agent in unhappy_agents:
            self.grid.move_to_empty(agent)

        return len(unhappy_agents)

    def run(self):
        for i in range(MAX_EPIDEMIC):
            if not self.is_there_epidemic():
                print("epidemic stopped at step", self.total_steps)
                self.saved_grids["final"] = self.get_array()
                break

            self.moved_last_step = 0
            self.spread_disease_synchronously()

            for agent in self.grid.agents:
                if agent.state == "infected":
                    agent.infected_steps += 1

                    if agent.infected_steps >= INFECT_DAYS:
                        agent.state = "resistant"

            if self.movement_enabled:
                self.moved_last_step = self.move_unhappy_agents()

            self.total_steps += 1
            self.update_stats()
            self.save()


In [ ]:
# Basic tests to see it working

s = schelling_model()
s.run()
print("Schelling final segregation:", s.final_seg_score)
print()

a = disease_spread_model(s, movement_enabled=False)
a.run()
print("Model A final segregation:", get_segregation_score(a.grid))
print("Model A total ever infected:", a.statistics_per_step[-1]["ever_infected"])
print("Model A total moves:", sum(stats["moved"] for stats in a.statistics_per_step))
print()

b = disease_spread_model(s, movement_enabled=True, initial_infected_positions_=a.initial_infected_positions)
b.run()
print("Model B final segregation:", get_segregation_score(b.grid))
print("Model B total ever infected:", b.statistics_per_step[-1]["ever_infected"])
print("Model B total moves:", sum(stats["moved"] for stats in b.statistics_per_step))
print()


# Run logic Statistical Data Analysis and helpers

In [ ]:
# Actual run logic over multiple seeds and paremeters and helper functions for generating plots later

STAT_SEEDS = [111, 222, 333, 444, 555]
STAT_WORKERS = min(4, len(STAT_SEEDS))
FEAR_SWEEP_VALUES = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
INITIAL_INFECTION_VARIANTS = [
    ("random_population", "Random population"),
    ("single_group", "One group only"),
    ("group_boundary", "Group boundaries"),
]
STATS_DIR = Path("stats")
STATS_DIR.mkdir(exist_ok=True)
STATS_SUMMARY_PATH = STATS_DIR / "stats_summary.csv"
MODEL_COLORS = {"Model A": "#1f70aa", "Model B": "#bd1515"}


def model_name(model):
    return "Model B" if model.movement_enabled else "Model A"


def model_time_series_df(model, seed, experiment="base", **extra):
    rows = []
    for stats in model.statistics_per_step:
        row = dict(stats)
        row.update(extra)
        row["seed"] = seed
        row["model"] = model_name(model)
        row["experiment"] = experiment
        row["initial_infection_type"] = model.initial_infection_type
        rows.append(row)
    return pd.DataFrame(rows)


def endpoint_summary(model, seed, experiment="base", **extra):
    df = pd.DataFrame(model.statistics_per_step)
    infected_steps = df[df["infected"] > 0]["step"]
    duration = int(infected_steps.max() - infected_steps.min() + 1) if len(infected_steps) else 0
    row = {
        "seed": seed,
        "experiment": experiment,
        "model": model_name(model),
        "initial_infection_type": model.initial_infection_type,
        "total_ever_infected": int(df["ever_infected"].max()),
        "peak_infected": int(df["infected"].max()),
        "epidemic_duration": duration,
        "final_segregation_score": get_segregation_score(model.grid),
        "mean_mixed_neighbor_agents": float(df["mixed_neighbor_agents"].mean()),
        "peak_cross_group_infected_neighbor_agents": int(df["cross_group_infected_neighbor_agents"].max()),
        "total_moves": int(df["moved"].sum()),
    }
    row.update(extra)
    return row


def run_pair_on_schelling(schelling, seed, initial_infection_type="random_population", b_coef=DEFAULT_B_COEF):
    model_a = disease_spread_model(
        schelling,
        seed=seed,
        movement_enabled=False,
        initial_infection_type=initial_infection_type,
        initial_infection_group=0,
    )
    model_a.run()

    model_b = disease_spread_model(
        schelling,
        seed=seed,
        movement_enabled=True,
        b_coef=b_coef,
        initial_infected_positions_=model_a.initial_infected_positions,
        initial_infection_type=initial_infection_type,
        initial_infection_group=0,
    )
    model_b.run()
    return model_a, model_b


def run_statistics_for_seed(seed):
    summaries = []
    time_frames = []
    schelling = schelling_model(seed=seed)
    schelling.run()

    model_a, model_b = run_pair_on_schelling(schelling, seed)
    for model in [model_a, model_b]:
        summaries.append(endpoint_summary(model, seed, "base"))
        time_frames.append(model_time_series_df(model, seed, "base"))

    for b_value in FEAR_SWEEP_VALUES:
        model_b_sweep = disease_spread_model(
            schelling,
            seed=seed,
            movement_enabled=True,
            b_coef=b_value,
            initial_infected_positions_=model_a.initial_infected_positions,
        )
        model_b_sweep.run()
        summaries.append(endpoint_summary(model_b_sweep, seed, "fear_sweep", b_coef=b_value))

    for infection_type, infection_label in INITIAL_INFECTION_VARIANTS:
        variant_a, variant_b = run_pair_on_schelling(schelling, seed, initial_infection_type=infection_type)
        for model in [variant_a, variant_b]:
            summaries.append(endpoint_summary(model, seed, "initial_infection_variant", initial_infection_label=infection_label))
            time_frames.append(model_time_series_df(model, seed, "initial_infection_variant", initial_infection_label=infection_label))

    return pd.DataFrame(summaries), pd.concat(time_frames, ignore_index=True)



def ci95(series):
    if series.count() <= 1:
        return 0.0
    return 1.96 * series.std(ddof=1) / np.sqrt(series.count())


def pad_time_series_for_plot(time_df, group_cols=None):
    if group_cols is None:
        group_cols = ["model"]
    padded_frames = []
    max_step = int(time_df["step"].max())
    columns_to_pad = [
        "infected",
        "ever_infected",
        "moved",
        "mixed_neighbor_agents",
        "cross_group_infected_neighbor_agents",
    ]
    columns_to_pad = [column for column in columns_to_pad if column in time_df.columns]

    for keys, group_df in time_df.groupby(["seed"] + group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        key_values = dict(zip(["seed"] + group_cols, keys))
        model_df = group_df.sort_values("step").set_index("step")
        model_df = model_df.reindex(range(max_step + 1)).ffill()
        model_df["step"] = model_df.index
        for column, value in key_values.items():
            model_df[column] = value
        after_stop = model_df["step"] > int(group_df["step"].max())
        for column in ["infected", "moved"]:
            if column in model_df.columns:
                model_df.loc[after_stop, column] = 0
        padded_frames.append(model_df.reset_index(drop=True)[["seed"] + group_cols + ["step"] + columns_to_pad])

    return pd.concat(padded_frames, ignore_index=True)


def time_series_stats(time_df, group_cols):
    rows = []
    metrics = ["infected", "ever_infected", "moved", "mixed_neighbor_agents", "cross_group_infected_neighbor_agents"]
    for keys, group_df in time_df.groupby(group_cols + ["step"]):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_cols + ["step"], keys))
        for metric in metrics:
            if metric in group_df.columns:
                row[f"{metric}_mean"] = group_df[metric].mean()
                row[f"{metric}_ci95"] = ci95(group_df[metric])
        rows.append(row)
    return pd.DataFrame(rows)


def endpoint_table(summary_df, group_cols, metrics):
    rows = []
    for keys, group_df in summary_df.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        key_values = dict(zip(group_cols, keys))
        for metric in metrics:
            row = dict(key_values)
            row["metric"] = metric
            row["mean"] = group_df[metric].mean()
            row["std"] = group_df[metric].std(ddof=1)
            rows.append(row)
    return pd.DataFrame(rows)


def plot_with_ci(ax, df, y, color, label, linestyle="-"):
    ax.plot(df["step"], df[f"{y}_mean"], color=color, linestyle=linestyle, label=label)
    ax.fill_between(
        df["step"],
        df[f"{y}_mean"] - df[f"{y}_ci95"],
        df[f"{y}_mean"] + df[f"{y}_ci95"],
        color=color,
        alpha=0.15,
        linewidth=0,
    )


In [ ]:
# Generate repeated-run data once.

with ThreadPool(STAT_WORKERS) as pool:
    repeated_results = pool.map(run_statistics_for_seed, STAT_SEEDS)

endpoint_summary_df = pd.concat([result[0] for result in repeated_results], ignore_index=True)
statistical_time_series_df = pd.concat([result[1] for result in repeated_results], ignore_index=True)

summary_metrics = [
    "total_ever_infected",
    "peak_infected",
    "epidemic_duration",
    "final_segregation_score",
    "mean_mixed_neighbor_agents",
    "peak_cross_group_infected_neighbor_agents",
    "total_moves",
]
stats_summary_df = pd.concat([
    endpoint_table(endpoint_summary_df[endpoint_summary_df["experiment"] == "base"], ["experiment", "model"], summary_metrics),
    endpoint_table(endpoint_summary_df[endpoint_summary_df["experiment"] == "fear_sweep"], ["experiment", "b_coef"], ["total_ever_infected", "epidemic_duration", "final_segregation_score", "total_moves"]),
    endpoint_table(endpoint_summary_df[endpoint_summary_df["experiment"] == "initial_infection_variant"], ["experiment", "initial_infection_label", "model"], ["total_ever_infected", "peak_infected", "epidemic_duration", "total_moves"]),
], ignore_index=True)
stats_summary_df.round(3).to_csv(STATS_SUMMARY_PATH, index=False)

time_series_df = statistical_time_series_df[statistical_time_series_df["experiment"] == "base"].copy()

print(f"Generated repeated-run data for {len(STAT_SEEDS)} seeds")
print(f"Wrote {STATS_SUMMARY_PATH}")
display(stats_summary_df.round(3))


# Schelling grid plots

In [ ]:
# Grid plots for schelling grids

FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)


def export_figure(fig, number, slug, **savefig_kwargs):
    path = FIGURE_DIR / f"plot_{number:02d}_{slug}.png"
    options = {"dpi": 200, "bbox_inches": "tight"}
    options.update(savefig_kwargs)
    fig.savefig(path, **options)
    print(f"Exported {path}")
    return path


GROUP_COLORS = ["#f2f2f2", "#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]
GROUP_CMAP = ListedColormap(GROUP_COLORS)
GROUP_NORM = BoundaryNorm(np.arange(-1.5, NUM_GROUPS + 0.5), len(GROUP_COLORS))


def show_grid_collection(items, max_cols=3):
    cols = min(max_cols, len(items))
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False, constrained_layout=True)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, item in zip(axes.ravel(), items):
        model = item["model"]
        step = item.get("step", "final")
        title = item.get("title")

        if step in model.saved_grids:
            grid_data = model.saved_grids[step]
            step_label = step
        else:
            grid_data = model.get_array()
            step_label = getattr(model, "total_steps", None)

        groups = np.full((len(grid_data), len(grid_data[0])), -1, dtype=int)
        states = np.full(groups.shape, None, dtype=object)
        for y, row in enumerate(grid_data):
            for x, cell in enumerate(row):
                if cell is not None:
                    groups[y, x] = cell[0]
                    states[y, x] = cell[1]

        ax.imshow(groups, cmap=GROUP_CMAP, norm=GROUP_NORM, origin="upper")

        infected_y, infected_x = np.where(states == "infected")
        if len(infected_x):
            ax.scatter(infected_x, infected_y, c="#d62728", marker="x", s=12, linewidths=0.8)

        if item.get("show_initial_infections", True) and hasattr(model, "initial_infected_positions"):
            initial_positions = [pos for pos in model.initial_infected_positions if pos is not None]
            if initial_positions:
                ax.scatter(
                    [pos[0] for pos in initial_positions],
                    [pos[1] for pos in initial_positions],
                    facecolors="none",
                    edgecolors="#ffd700",
                    marker="o",
                    s=14,
                    linewidths=0.8,
                )

        if title is None:
            title = model.__class__.__name__
        ax.set_title(f"{title} | step {step_label}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    return fig


def show_grid_sequence(model, steps=None, title_prefix=None, show_initial_infections=True, max_cols=5):
    if steps is None:
        steps = sorted(step for step in model.saved_grids if isinstance(step, (int, np.integer)))
        if "final" in model.saved_grids:
            steps.append("final")
    else:
        steps = [step for step in steps if step == "final" or step in model.saved_grids]

    items = [
        {
            "model": model,
            "step": step,
            "title": title_prefix,
            "show_initial_infections": show_initial_infections,
        }
        for step in steps
    ]
    return show_grid_collection(items, max_cols=max_cols)


In [ ]:
fig = show_grid_collection([
    {"model": s, "step": "final", "title": "Schelling final", "show_initial_infections": False},
    {"model": a, "step": "final", "title": "Disease model A final", "show_initial_infections": True},
    {"model": b, "step": "final", "title": "Disease model B final", "show_initial_infections": True},
], max_cols=3)
export_figure(fig, 1, "final_grid_group")
plt.show()

In [ ]:
# show grids for the variety of different models
fig = show_grid_sequence(s, steps=[0, 25, 50, 75, "final"], title_prefix="Schelling separation", show_initial_infections=False)
export_figure(fig, 2, "schelling_separation_sequence")
plt.show()

fig = show_grid_sequence(a, steps=[0, 25, 50, 75, "final"], title_prefix="Disease model A")
export_figure(fig, 3, "disease_model_a_sequence")
plt.show()

fig = show_grid_sequence(b, steps=[0, 25, 50, 75, "final"], title_prefix="Disease model B")
export_figure(fig, 4, "disease_model_b_sequence")
plt.show()

# Time Analysis

In [ ]:
# Time Analysis plots. Re-run this cell to edit/regenerate plots 05-08 without re-running simulations.

base_plot_df = pad_time_series_for_plot(time_series_df, ["model"])
base_stats_df = time_series_stats(base_plot_df, ["model"])

# Epidemic curves
fig, ax = plt.subplots(figsize=(9, 5))
for model, color in MODEL_COLORS.items():
    model_df = base_stats_df[base_stats_df["model"] == model].sort_values("step")
    plot_with_ci(ax, model_df, "infected", color, f"{model} active infected")
    plot_with_ci(ax, model_df, "ever_infected", color, f"{model} cumulative ever-infected", linestyle="--")
ax.set_title("Epidemic curves, mean with 95% CI")
ax.set_xlabel("Step")
ax.set_ylabel("Agents")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
export_figure(fig, 5, "epidemic_curves")
plt.show()

# Movement and infection over time
fig, ax = plt.subplots(figsize=(9, 5))
for model, color in MODEL_COLORS.items():
    model_df = base_stats_df[base_stats_df["model"] == model].sort_values("step")
    plot_with_ci(ax, model_df, "infected", color, f"{model} active infected")
model_b_df = base_stats_df[base_stats_df["model"] == "Model B"].sort_values("step")
plot_with_ci(ax, model_b_df, "moved", "#fb6a4a", "Model B moved agents per step")
ax.set_title("Movement and infection over time, mean with 95% CI")
ax.set_xlabel("Step")
ax.set_ylabel("Agents")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
export_figure(fig, 6, "movement_and_infection_model_b")
plt.show()

# Mixing and infection over time
MIXING_PLOT_COLORS = {
    "Model A": {
        "infected": "#08306b",
        "mixed_neighbor_agents": "#2171b5",
        "cross_group_infected_neighbor_agents": "#6fc4cf",
    },
    "Model B": {
        "infected": "#67000d",
        "mixed_neighbor_agents": "#cb181d",
        "cross_group_infected_neighbor_agents": "#fb6a4a",
    },
}
for plot_number, model in enumerate(["Model A", "Model B"], start=7):
    colors = MIXING_PLOT_COLORS[model]
    model_df = base_stats_df[base_stats_df["model"] == model].sort_values("step")
    fig, ax = plt.subplots(figsize=(9, 5))
    plot_with_ci(ax, model_df, "infected", colors["infected"], "Active infected")
    plot_with_ci(ax, model_df, "mixed_neighbor_agents", colors["mixed_neighbor_agents"], "Agents with different-group neighbor")
    plot_with_ci(ax, model_df, "cross_group_infected_neighbor_agents", colors["cross_group_infected_neighbor_agents"], "Agents with different-group infected neighbor")
    ax.set_title(f"Mixing and infection over time, {model}, mean with 95% CI")
    ax.set_xlabel("Step")
    ax.set_ylabel("Agents")
    ax.legend(frameon=False)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    export_figure(fig, plot_number, f"mixing_and_infection_{model.lower().replace(' ', '_')}")
    plt.show()


## Fear Parameter Sweep


In [ ]:
# Fear parameter sweep plot and endpoint summary.

sweep_df = endpoint_summary_df[endpoint_summary_df["experiment"] == "fear_sweep"]
sweep_metrics = ["total_ever_infected", "epidemic_duration", "final_segregation_score", "total_moves"]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
for ax, metric in zip(axes.ravel(), sweep_metrics):
    grouped = sweep_df.groupby("b_coef")[metric]
    mean = grouped.mean()
    band = grouped.apply(ci95)
    plot_df = pd.DataFrame({
        "step": mean.index,
        f"{metric}_mean": mean.values,
        f"{metric}_ci95": band.values,
    })
    plot_with_ci(ax, plot_df, metric, MODEL_COLORS["Model B"], metric.replace("_", " "))
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("Fear factor b")
    ax.grid(alpha=0.2)
fig.suptitle("Fear parameter sweep, mean with 95% CI")
export_figure(fig, 9, "fear_factor_sweep")
plt.show()

sweep_summary_df = stats_summary_df[
    stats_summary_df["experiment"] == "fear_sweep"
].copy()
display(sweep_summary_df.round(3))


# Initial Infection Analysis

In [ ]:
# Show grid sequences for the two new starting infection types.

GRID_SEQUENCE_SEED = STAT_SEEDS[0]
grid_sequence_steps = [0, 25, 50, 75, "final"]
grid_sequence_variants = [
    variant for variant in INITIAL_INFECTION_VARIANTS
    if variant[0] in ["single_group", "group_boundary"]
]

for plot_number, (infection_type, infection_label) in enumerate(grid_sequence_variants, start=10):
    print(f"Grid sequence for {infection_label}, seed {GRID_SEQUENCE_SEED}")
    schelling = schelling_model(seed=GRID_SEQUENCE_SEED)
    schelling.run()
    model_a, model_b = run_pair_on_schelling(
        schelling,
        GRID_SEQUENCE_SEED,
        initial_infection_type=infection_type,
    )

    fig = show_grid_sequence(
        model_b,
        steps=grid_sequence_steps,
        title_prefix=f"Model B, {infection_label}",
        show_initial_infections=True,
        max_cols=5,
    )
    export_figure(fig, plot_number, f"starting_infection_grid_sequence_model_b_{infection_type}")
    plt.show()


In [ ]:
# Starting infection variants plot and endpoint summary.

variant_df = statistical_time_series_df[statistical_time_series_df["experiment"] == "initial_infection_variant"]
variant_plot_df = pad_time_series_for_plot(variant_df, ["initial_infection_type", "initial_infection_label", "model"])
variant_stats_df = time_series_stats(variant_plot_df, ["initial_infection_type", "initial_infection_label", "model"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for ax, (infection_type, infection_label) in zip(axes, INITIAL_INFECTION_VARIANTS):
    variant_stats = variant_stats_df[variant_stats_df["initial_infection_type"] == infection_type]
    for model, color in MODEL_COLORS.items():
        model_df = variant_stats[variant_stats["model"] == model].sort_values("step")
        plot_with_ci(ax, model_df, "infected", color, f"{model} active")
        plot_with_ci(ax, model_df, "ever_infected", color, f"{model} cumulative", linestyle="--")
    ax.set_title(infection_label)
    ax.set_xlabel("Step")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Agents")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=12, loc="lower center", ncol=4)
fig.suptitle("Epidemic curves by starting infection placement, mean with 95% CI")
fig.tight_layout(rect=[0, 0.12, 1, 0.94])
export_figure(fig, 12, "starting_infection_variant_epidemic_curves")
plt.show()

variant_endpoint_table = stats_summary_df[
    stats_summary_df["experiment"] == "initial_infection_variant"
].copy()
display(variant_endpoint_table.round(3))
